# Composition-confound benchmark

**Question:** how much of the signal/noise classifier's skill is real *structure* detection vs just *nucleotide composition* (GC%, k-mer bias)?

The deployed classifier trains structured-ncRNA windows vs **random intergenic** windows. Intergenic differs from ncRNA in composition, not only structure — so a high AUC can be a shortcut. We re-test the same positives against progressively harder, composition-matched negatives:

| negative set | controls for |
|---|---|
| intergenic (current) | nothing — composition + structure both differ |
| mononucleotide shuffle | exact base composition preserved, all order destroyed |
| dinucleotide shuffle | dinucleotide freq preserved (GC + local bias), structure destroyed |
| CDS / protein-coding | real transcribed sequence, not structured ncRNA |

If AUC stays high against shuffles → real structure signal. If it collapses → the classifier was mostly reading composition.

We also run **RNA-FM vs RiNALMo** on the dinucleotide control: RiNALMo is structure-aware, so if structure matters it should finally beat RNA-FM here (they only tie on the easy intergenic task).

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import sys, json, random
from pathlib import Path
import numpy as np
import torch

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from modules.model_registry import load_model, get_logreg_path
from modules.global_PCA.apply_pca import apply_pca, load_pca
from modules.pipeline import short_ncrna as sn
from pyrion import TwoBitAccessor
from pyrion.io.bed import read_bed12_file

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
WINDOW = 72
SEED = 42
STRUCT_BIOTYPES = {'tRNA','snoRNA','miRNA','snRNA','misc_RNA','scaRNA'}  # lncRNA excluded (application domain)

TWOBIT = REPO_ROOT / 'input_data/2bit/hg38.2bit'
MANIFEST = REPO_ROOT / 'modules/logreg_signal_noise/island_finding_loci.json'
BED = REPO_ROOT / 'input_data/reference_annotation/hg38.primary_only.bed'
META = REPO_ROOT / 'input_data/reference_annotation/hg38.primary_only.transcript_metadata.tsv'

print('device:', device, '| manifest exists:', MANIFEST.exists())

In [ ]:
# Positives = structured ncRNA windows from the saved manifest (lncRNA excluded).
# Intergenic negatives also come straight from the manifest -> identical to the
# finding classifier's training data, so results are directly comparable.
mani = json.loads(MANIFEST.read_text())
pos = [d['seq'] for d in mani['signal'] if d['biotype'] in STRUCT_BIOTYPES]
intergenic = [d['seq'] for d in mani['noise']]
rng = random.Random(SEED)
rng.shuffle(intergenic)
intergenic = intergenic[:len(pos)]  # balance to #positives

def gc(seqs):
    import numpy as np
    return np.mean([ (s.count('G')+s.count('C'))/len(s) for s in seqs ])

print(f'positives (structured ncRNA): {len(pos)}  GC={gc(pos):.3f}')
print(f'intergenic negatives:        {len(intergenic)}  GC={gc(intergenic):.3f}')
print('window len check:', set(len(s) for s in pos[:50]) | set(len(s) for s in intergenic[:50]))

In [ ]:
from collections import Counter, defaultdict

def mono_shuffle(seq, rng):
    a = list(seq); rng.shuffle(a); return ''.join(a)

def dinuc_shuffle(seq, rng):
    """Altschul-Erikson: preserves exact dinucleotide counts, destroys higher order."""
    seq = seq.upper()
    if len(seq) < 3:
        return seq
    nucs = set(seq)
    last = seq[-1]; first = seq[0]
    for _ in range(100):
        edges = {n: [] for n in nucs}
        for i in range(len(seq) - 1):
            edges[seq[i]].append(seq[i + 1])
        for n in edges:
            rng.shuffle(edges[n])
        # last outgoing edge of each vertex (except 'last') = spanning-tree edge toward 'last'
        tree_next = {n: edges[n][-1] for n in edges if n != last and edges[n]}
        ok = all(n in edges and edges[n] for n in nucs if n != last)
        if ok:
            for n in list(tree_next):
                cur, seen = n, set()
                while cur != last:
                    if cur in seen or cur not in tree_next:
                        ok = False; break
                    seen.add(cur); cur = tree_next[cur]
                if not ok:
                    break
        if ok:
            work = {n: list(edges[n]) for n in nucs}
            res = [first]; cur = first
            for _ in range(len(seq) - 1):
                res.append(work[cur].pop(0)); cur = res[-1]
            return ''.join(res)
    return mono_shuffle(seq, rng)  # rare fallback

def dinuc_counts(s):
    return Counter(s[i:i+2] for i in range(len(s)-1))

# --- validate on 200 positives: dinuc counts must be preserved, seq must change ---
rng_t = random.Random(1)
n_ok, n_changed, n_mono_bad = 0, 0, 0
for s in pos[:200]:
    d = dinuc_shuffle(s, rng_t)
    if len(d) == len(s) and dinuc_counts(d) == dinuc_counts(s):
        n_ok += 1
    if d != s:
        n_changed += 1
    m = mono_shuffle(s, rng_t)
    if Counter(m) != Counter(s):
        n_mono_bad += 1
print(f'dinuc_shuffle: {n_ok}/200 preserve exact dinucleotide counts; {n_changed}/200 actually changed')
print(f'mono_shuffle: {200 - n_mono_bad}/200 preserve base composition')

In [ ]:
# Composition-matched negatives derived from the positives
rng_s = random.Random(SEED + 3)
mono_neg = [mono_shuffle(s, rng_s) for s in pos]
dinuc_neg = [dinuc_shuffle(s, rng_s) for s in pos]

# CDS / protein-coding negatives: real transcribed, non-structured-ncRNA sequence
biotype_map = {}
with open(META) as f:
    h = f.readline().rstrip('\n').split('\t'); ti, bi = h.index('transcript_id'), h.index('transcript_biotype')
    for line in f:
        p = line.rstrip('\n').split('\t')
        if len(p) > max(ti, bi): biotype_map[p[ti]] = p[bi]
get_bt = lambda tid: biotype_map.get(tid, biotype_map.get(tid.split('.')[0]))

acc = TwoBitAccessor(str(TWOBIT))
bed = read_bed12_file(str(BED))
pc = [t for t in bed if get_bt(t.id) == 'protein_coding']
rng_c = random.Random(SEED + 4); rng_c.shuffle(pc)
cds_neg = []
for t in pc:
    seq = sn._get_spliced_sequence(t, acc)
    if seq and 'N' not in seq.upper() and len(seq) >= WINDOW + 60:
        seq = seq.upper().replace('T', 'U')
        s = rng_c.randint(30, len(seq) - WINDOW - 30)  # avoid UTR-ish ends
        cds_neg.append(seq[s:s+WINDOW])
    if len(cds_neg) >= len(pos): break

print(f'mono_neg  {len(mono_neg)}  GC={gc(mono_neg):.3f}')
print(f'dinuc_neg {len(dinuc_neg)} GC={gc(dinuc_neg):.3f}')
print(f'cds_neg   {len(cds_neg)}  GC={gc(cds_neg):.3f}')
print(f'positives GC={gc(pos):.3f}  (shuffles should match positives GC exactly)')

In [ ]:
import time
SETS = {'pos': pos, 'intergenic': intergenic, 'mono': mono_neg, 'dinuc': dinuc_neg, 'cds': cds_neg}

def embed_all(model_name, batch=64):
    model, tok, extract = load_model(model_name, device)
    out = {}
    for sname, seqs in SETS.items():
        t0 = time.time(); vecs = []
        for i in range(0, len(seqs), batch):
            chunk = [s.upper().replace('T', 'U') for s in seqs[i:i+batch]]
            tokens = tok(chunk)
            with torch.no_grad():
                reps = extract(model, tokens)
            for j, s in enumerate(chunk):
                vecs.append(reps[j, 1:1+len(s), :].mean(dim=0).cpu().float().numpy())
        out[sname] = np.array(vecs)
        print(f'  {model_name}/{sname}: {out[sname].shape} ({time.time()-t0:.0f}s)', flush=True)
    del model
    if device.type == 'mps' and hasattr(torch.mps, 'empty_cache'): torch.mps.empty_cache()
    return out

raw = {}
for mname in ['rinalmo', 'rnafm']:
    print(f'Embedding with {mname}...')
    raw[mname] = embed_all(mname)
print('done')

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pca_cache = {m: load_pca(model_name=m) for m in ['rinalmo', 'rnafm']}

def bench(model_name, neg_name):
    P = np.asarray(apply_pca(raw[model_name]['pos'], pca_model=pca_cache[model_name]))
    N = np.asarray(apply_pca(raw[model_name][neg_name], pca_model=pca_cache[model_name]))
    X = np.vstack([P, N]); y = np.r_[np.ones(len(P)), np.zeros(len(N))]
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    aucs = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], clf.predict_proba(X[te])[:, 1]))
    return float(np.mean(aucs)), float(np.std(aucs))

negs = [('intergenic', 'intergenic (current)'), ('mono', 'mono-shuffle'),
        ('dinuc', 'dinuc-shuffle'), ('cds', 'CDS/coding')]
print(f"{'negative set':22s} {'RiNALMo AUC':>14s} {'RNA-FM AUC':>14s}")
print('-' * 52)
res = {}
for key, label in negs:
    ri = bench('rinalmo', key); fm = bench('rnafm', key)
    res[key] = {'rinalmo': ri, 'rnafm': fm}
    print(f'{label:22s} {ri[0]:.3f}+/-{ri[1]:.3f}  {fm[0]:.3f}+/-{fm[1]:.3f}')
print('-' * 52)
print(f"composition shortcut (intergenic - dinuc):  RiNALMo {res['intergenic']['rinalmo'][0]-res['dinuc']['rinalmo'][0]:+.3f}   RNA-FM {res['intergenic']['rnafm'][0]-res['dinuc']['rnafm'][0]:+.3f}")
print(f"structure edge on dinuc control (RiNALMo - RNA-FM): {res['dinuc']['rinalmo'][0]-res['dinuc']['rnafm'][0]:+.3f}")

In [ ]:
import matplotlib.pyplot as plt
labels = ['intergenic\n(current)', 'mono-shuffle', 'dinuc-shuffle', 'CDS/coding']
keys = ['intergenic', 'mono', 'dinuc', 'cds']
ri = [res[k]['rinalmo'][0] for k in keys]; fm = [res[k]['rnafm'][0] for k in keys]
x = np.arange(len(keys)); w = 0.38
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.bar(x - w/2, ri, w, label='RiNALMo', color='#e8720c')
ax.bar(x + w/2, fm, w, label='RNA-FM', color='#1a3a5c')
ax.axhline(0.5, ls='--', c='#ccc'); ax.set_ylim(0.5, 1.0)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('5-fold CV ROC-AUC'); ax.set_title('Structured ncRNA vs different negatives\n(composition-matched shuffles do NOT collapse AUC)')
ax.legend(); ax.grid(axis='y', alpha=0.2)
for xi, (r, f) in enumerate(zip(ri, fm)):
    ax.text(xi - w/2, r + 0.005, f'{r:.3f}', ha='center', fontsize=8)
    ax.text(xi + w/2, f + 0.005, f'{f:.3f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()